Load the MNIST dataset (introduced in Chapter 3), and split it into a training set,
a validation set, and a test set (e.g., use 50,000 instances for training, 10,000 for
validation, and 10,000 for testing). Then train various classifiers, such as a
random forest classifier, an extra-trees classifier, and an SVM classifier. Next, try
to combine them into an ensemble that outperforms each individual classifier on
the validation set, using soft or hard voting. Once you have found one, try it on
the test set. How much better does it perform compared to the individual
classifiers?

In [1]:
from sklearn.datasets import fetch_openml
# 1) Load MNIST
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype("int64")
# 2) Split into train/val/test: 50k / 10k / 10k
X_train, y_train = X[:50000], y[:50000]
X_val, y_val = X[50000:60000], y[50000:60000]
X_test, y_test = X[60000:], y[60000:]
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (50000, 784) y_train: (50000,)
X_val: (10000, 784) y_val: (10000,)
X_test: (10000, 784) y_test: (10000,)


Random Forest

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
# Train Random Forest
rf_clf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)
# Validation accuracy
y_val_pred = rf_clf.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)
print(f"Random Forest validation accuracy: {val_acc:.4f}")

Random Forest validation accuracy: 0.9718


Extra Trees

In [3]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score

# Train Extra Trees
extra_trees_clf = ExtraTreesClassifier(random_state=42, n_jobs=-1)
extra_trees_clf.fit(X_train, y_train)

# Validation accuracy
y_val_pred_extra = extra_trees_clf.predict(X_val)
val_acc_extra = accuracy_score(y_val, y_val_pred_extra)

print(f"Extra Trees validation accuracy: {val_acc_extra:.4f}")

Extra Trees validation accuracy: 0.9757


SVM

In [4]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
# Train SVM (RBF kernel)
svm_clf = SVC(kernel="rbf", gamma="scale", C=1.0, random_state=42)
svm_clf.fit(X_train, y_train)
# Validation accuracy
y_val_pred_svm = svm_clf.predict(X_val)
val_acc_svm = accuracy_score(y_val, y_val_pred_svm)
print(f"SVM validation accuracy: {val_acc_svm:.4f}")

SVM validation accuracy: 0.9802


Combination model hard and soft voting

In [5]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
# Reuse already-trained model configs (fresh instances for voting)
rf_vote = RandomForestClassifier(random_state=42, n_jobs=-1)
et_vote = ExtraTreesClassifier(random_state=42, n_jobs=-1)
sv_vote = SVC(random_state=42)
# Hard voting ensemble
voting_hard = VotingClassifier(
    estimators=[
        ("rf", rf_vote),
        ("et", et_vote),
        ("gb", sv_vote),
    ],
    voting="hard"
)
voting_hard.fit(X_train, y_train)
y_val_pred_hard = voting_hard.predict(X_val)
val_acc_hard = accuracy_score(y_val, y_val_pred_hard)
print(f"Hard Voting validation accuracy: {val_acc_hard:.4f}")
# Soft voting ensemble
voting_soft = VotingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=42, n_jobs=-1)),
        ("et", ExtraTreesClassifier(random_state=42, n_jobs=-1)),
        ("sv", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)),
    ],
    voting="soft"
)
voting_soft.fit(X_train, y_train)
y_val_pred_soft = voting_soft.predict(X_val)
val_acc_soft = accuracy_score(y_val, y_val_pred_soft)
print(f"Soft Voting validation accuracy: {val_acc_soft:.4f}")

Hard Voting validation accuracy: 0.9772
Soft Voting validation accuracy: 0.9814


Run the individual classifiers from the previous exercise to make predictions on
the validation set, and create a new training set with the resulting predictions:
each training instance is a vector containing the set of predictions from all your
classifiers for an image, and the target is the image’s class. Train a classifier on
this new training set. Congratulations—you have just trained a blender, and
together with the classifiers it forms a stacking ensemble! Now evaluate the
ensemble on the test set. For each image in the test set, make predictions with all
your classifiers, then feed the predictions to the blender to get the ensemble’s
predictions. How does it compare to the voting classifier you trained earlier?
Now try again using a StackingClassifier instead. Do you get better
performance? If so, why?
Solutions to these exercises are avail

Stacking Classifier:

Trains base models
Gets their predictions (via cross-validation)
Uses those predictions as features
Trains a meta-model (blender) on top

Without Classifier:

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
# Base model predictions on validation set (meta-training data)
rf_val_pred = rf_clf.predict(X_val)
et_val_pred = extra_trees_clf.predict(X_val)
gb_val_pred = gb_clf.predict(X_val)
X_meta_train = np.c_[rf_val_pred, et_val_pred, gb_val_pred]
y_meta_train = y_val
# Train meta-learner
meta_clf = LogisticRegression(max_iter=1000, random_state=42)
meta_clf.fit(X_meta_train, y_meta_train)
# Build meta-test data from base model predictions on test set
rf_test_pred = rf_clf.predict(X_test)
et_test_pred = extra_trees_clf.predict(X_test)
gb_test_pred = gb_clf.predict(X_test)
X_meta_test = np.c_[rf_test_pred, et_test_pred, gb_test_pred]
meta_test_pred = meta_clf.predict(X_meta_test)
print("Manual stacking test accuracy:", accuracy_score(y_test, meta_test_pred))

NameError: name 'gb_clf' is not defined

With StackingClassifier

In [ ]:
# 2) StackingClassifier (separate implementation)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

stack_clf = StackingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(random_state=42, n_jobs=-1)),
        ("et", ExtraTreesClassifier(random_state=42, n_jobs=-1)),
        ("gb", GradientBoostingClassifier(random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

stack_clf.fit(X_train, y_train)
stack_test_pred = stack_clf.predict(X_test)

print("StackingClassifier test accuracy:", accuracy_score(y_test, stack_test_pred))